In [16]:
import pandas as pd
import numpy as np
import torch
import gc
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer, 
    AutoModelForQuestionAnswering, 
    TrainingArguments, 
    Trainer,
    DefaultDataCollator
)
from sentence_transformers import SentenceTransformer, util
from tqdm import tqdm

In [17]:
MODEL_NAME = "microsoft/deberta-v3-base" 
OUTPUT_DIR = "./deberta_sarang_plus"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"🚀 Using Device: {DEVICE}")

🚀 Using Device: cuda


In [18]:
print("\n" + "="*40)
print("🧹 PHASE 1: Strict Data Filtering")
print("="*40)

def load_filtered_data(file_path):
    df = pd.read_csv(file_path, sep=";", engine="python")
    clean_samples = []
    skipped_count = 0
    
    for _, row in df.dropna(subset=["Text", "Question"]).iterrows():
        # Evaluation data usually has no Answer column
        if "Answer" not in df.columns:
             clean_samples.append({
                "id": str(row["ID"]),
                "context": row["Text"],
                "question": row["Question"],
                "answers": {"text": [], "answer_start": []}
            })
             continue

        # TRAINING DATA: Apply Filter
        ans = str(row["Answer"]).strip()
        context = str(row["Text"])
        
        # KEY STEP: Find exact position. If missing, DISCARD.
        pos = context.find(ans)
        
        if pos != -1:
            clean_samples.append({
                "id": str(row["ID"]),
                "context": context,
                "question": row["Question"],
                "answers": {"text": [ans], "answer_start": [pos]}
            })
        else:
            skipped_count += 1
            
    print(f"📉 Filtered out {skipped_count} 'dirty' examples.")
    print(f"✅ Kept {len(clean_samples)} 'clean' extractive examples.")
    return Dataset.from_list(clean_samples)


🧹 PHASE 1: Strict Data Filtering


In [19]:
# Load Data
train_dataset = load_filtered_data("training_data_en.csv")
eval_dataset = load_filtered_data("evaluation_data_en.csv")

# Create Validation Split
split = train_dataset.train_test_split(test_size=0.1, seed=42)
dataset = DatasetDict({"train": split["train"], "validation": split["test"]})

📉 Filtered out 13 'dirty' examples.
✅ Kept 1986 'clean' extractive examples.
📉 Filtered out 0 'dirty' examples.
✅ Kept 498 'clean' extractive examples.


In [20]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def preprocess_function(examples):
    inputs = tokenizer(
        examples["question"],
        examples["context"],
        max_length=384,
        truncation="only_second",
        stride=128,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    offset_mapping = inputs.pop("offset_mapping")
    sample_map = inputs.pop("overflow_to_sample_mapping")
    start_positions = []
    end_positions = []

    for i, offsets in enumerate(offset_mapping):
        sample_idx = sample_map[i]
        answers = examples["answers"][sample_idx]
        
        # If no answer (e.g., eval set), set to CLS
        if len(answers["answer_start"]) == 0:
            start_positions.append(0)
            end_positions.append(0)
            continue

        start_char = answers["answer_start"][0]
        end_char = start_char + len(answers["text"][0])
        
        sequence_ids = inputs.sequence_ids(i)
        
        # Find context token span
        idx = 0
        while sequence_ids[idx] != 1: idx += 1
        context_start = idx
        while sequence_ids[idx] == 1: idx += 1
        context_end = idx - 1

        # Check if answer is inside context span
        if offsets[context_start][0] > start_char or offsets[context_end][1] < end_char:
            start_positions.append(0)
            end_positions.append(0)
        else:
            idx = context_start
            while idx <= context_end and offsets[idx][0] <= start_char:
                idx += 1
            start_positions.append(idx - 1)

            idx = context_end
            while idx >= context_start and offsets[idx][1] >= end_char:
                idx -= 1
            end_positions.append(idx + 1)

    inputs["start_positions"] = start_positions
    inputs["end_positions"] = end_positions
    return inputs

tokenized_datasets = dataset.map(preprocess_function, batched=True, remove_columns=dataset["train"].column_names)

c:\Users\kisho\AppData\Local\Programs\Python\Python312\Lib\site-packages\transformers\convert_slow_tokenizer.py:564: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(
Map: 100%|██████████| 199/199 [00:00<00:00, 5466.84 examples/s]


In [21]:
print("\n" + "="*40)
print("🎯 PHASE 2: DeBERTa Fine-Tuning")
print("="*40)

model = AutoModelForQuestionAnswering.from_pretrained(MODEL_NAME)

args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,               # DeBERTa prefers lower LR
    per_device_train_batch_size=4,    # Smaller batch for stability
    gradient_accumulation_steps=2,
    num_train_epochs=10,              # Enough for filtered data
    weight_decay=0.01,
    fp16=True,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    save_total_limit=1,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    tokenizer=tokenizer,
    data_collator=DefaultDataCollator(),
)

trainer.train()
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("✅ Training Complete.")


🎯 PHASE 2: DeBERTa Fine-Tuning


Some weights of DebertaV2ForQuestionAnswering were not initialized from the model checkpoint at microsoft/deberta-v3-base and are newly initialized: ['qa_outputs.bias', 'qa_outputs.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
C:\Users\kisho\AppData\Local\Temp\ipykernel_5148\3711781331.py:23: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [ ]:
print("\n" + "="*40)
print("🧠 PHASE 3: Inference with Purpose-Filtering")
print("="*40)

def predict_with_logic(context, question):
    inputs = tokenizer(question, context, return_tensors="pt", max_length=384, truncation="only_second").to(DEVICE)
    
    with torch.no_grad():
        outputs = model(**inputs)

    # Get Top-5 Candidates
    start_logits = outputs.start_logits[0]
    end_logits = outputs.end_logits[0]
    
    start_indexes = torch.argsort(start_logits, descending=True)[:5]
    end_indexes = torch.argsort(end_logits, descending=True)[:5]

    best_score = -float("inf")
    best_answer = ""
    
    # Paper-Based Guardrails 
    # Penalize "Purpose" markers unless question asks "Why/Purpose"
    purpose_markers = ["in order to", "aiming to", "with the goal", "to ensure"]
    
    for start_index in start_indexes:
        for end_index in end_indexes:
            if start_index > end_index: continue
            if end_index - start_index > 30: continue # Answers shouldn't be paragraphs
            
            # Score
            score = start_logits[start_index] + end_logits[end_index]
            
            # Extract Text
            ans_ids = inputs.input_ids[0][start_index : end_index + 1]
            ans_text = tokenizer.decode(ans_ids, skip_special_tokens=True).strip()
            
            # LOGIC RULE: If answer is "Purpose" but question isn't, penalize heavily
            if any(m in ans_text.lower() for m in purpose_markers):
                if "purpose" not in question.lower() and "why" not in question.lower():
                    score -= 5.0  # Heavy Penalty
            
            if score > best_score:
                best_score = score
                best_answer = ans_text
                
    return best_answer

# Evaluate
print("Generating Predictions...")
val_data = dataset["validation"]
preds = [predict_with_logic(x["context"], x["question"]) for x in tqdm(val_data)]
truths = [x["answers"]["text"][0] for x in val_data]

# Metrics
sas_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
em = sum([1 if p.strip() == t.strip() else 0 for p, t in zip(preds, truths)]) / len(truths)

with torch.no_grad():
    t_emb = sas_model.encode(truths, convert_to_tensor=True)
    p_emb = sas_model.encode(preds, convert_to_tensor=True)
    sas = torch.diag(util.cos_sim(t_emb, p_emb)).mean().item()

print(f"\n✅ Exact Match (EM): {em*100:.2f}%")
print(f"🧠 Semantic Score (SAS): {sas:.4f}")


🧠 PHASE 3: Inference with Purpose-Filtering
Generating Predictions...


100%|██████████| 199/199 [00:06<00:00, 29.15it/s]



✅ Exact Match (EM): 77.89%
🧠 Semantic Score (SAS): 0.9522
